In [2]:
from transformers import (
    BertForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    pipeline,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
)
from datasets import Dataset, load_dataset
from nltk.tokenize import word_tokenize

In [866]:
import pandas as pd

df = pd.read_csv("D:\\database\\csv files\\Restaurants_Train_v2.csv")
df2 = pd.read_csv("D:\\database\\csv files\\Laptop_Train_v2.csv")
df3 = pd.read_csv("D:\\database\\csv files\\Laptop_Train_v2.csv")

c_df = pd.concat([df, df2, df3])

In [867]:
c_df["Sentence"].drop_duplicates(inplace=True)

In [868]:
import re
import string


def preprocess(text):
    text = str(text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = re.sub(r"[@#]", "", text)
    return text.translate(str.maketrans("", "", string.punctuation))


c_df["Sentence"] = c_df["Sentence"].apply(preprocess)

In [869]:
grouped = c_df.groupby("Sentence")

In [105]:
def convert_to_bio(sentence, aspects):
    words = sentence.split()
    labels = ["O"] * len(words)

    # Track word positions
    current_pos = 0
    word_spans = []

    for word in words:
        start = sentence.find(word, current_pos)
        end = start + len(word)
        word_spans.append((start, end))
        current_pos = end

    # Mark BIO labels
    for aspect in aspects:
        aspect_start = aspect["from"]
        aspect_end = aspect["to"]

        for i, (word_start, word_end) in enumerate(word_spans):
            if word_start >= aspect_start and word_end <= aspect_end:
                if word_start == aspect_start:
                    labels[i] = "B-ASP"
                else:
                    labels[i] = "I-ASP"

    return words, labels

In [902]:
dataset = []

for sentence, group in grouped:
    aspects = group[["Aspect Term", "from", "to"]].to_dict("records")

    words, labels = convert_to_bio(sentence, aspects)

    dataset.append({"tokens": words, "labels": labels})

In [144]:
label2id = {"O": 0, "B-ASP": 1, "I-ASP": 2}

id2label = {v: k for k, v in label2id.items()}

In [498]:
model = AutoModelForTokenClassification.from_pretrained(
    "D:\\sentiment-analysis\\Multilang-sentiment-analysis\\aspect-token-clf",
)
tokenizer = AutoTokenizer.from_pretrained(
    "D:\\sentiment-analysis\\Multilang-sentiment-analysis\\aspect-token-clf"
)

In [929]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=128,
    )

    word_ids = tokenized_inputs.word_ids()
    labels = example["labels"]

    aligned_labels = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label2id[labels[word_idx]])
        else:
            # For subword tokens
            aligned_labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [148]:
train_dataset = Dataset.from_list(dataset)
# test_dataset = Dataset.from_pandas(df3)
tokenized_dataset = train_dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/3501 [00:00<?, ? examples/s]

In [251]:
training_args = TrainingArguments(
    output_dir="./aspect_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

C:\Users\pritc\AppData\Local\Temp\ipykernel_27456\2561802497.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=657, training_loss=0.2170834011501736, metrics={'train_runtime': 1344.5378, 'train_samples_per_second': 7.812, 'train_steps_per_second': 0.489, 'total_flos': 36865421104896.0, 'train_loss': 0.2170834011501736, 'epoch': 3.0})

In [499]:
pip = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)
pip(
    """The entire sentence reflects the "Service" category because it is an action taken by a service provider"""
)

Device set to use cpu
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


[{'entity_group': 'ASP',
  'score': 0.7143085,
  'word': 'sentence',
  'start': 11,
  'end': 19},
 {'entity_group': 'ASP',
  'score': 0.8825191,
  'word': 'service',
  'start': 34,
  'end': 41}]

In [39]:
text_model = BertForSequenceClassification.from_pretrained(
    "D:\\sentiment-analysis\\Multilang-sentiment-analysis\\aspect-sentimentV2-clf",
)
text_tokenizer = AutoTokenizer.from_pretrained(
    "D:\\sentiment-analysis\\Multilang-sentiment-analysis\\aspect-sentimentV2-clf"
)

In [715]:
text_model.config.id2label = {0: "negative", 1: "neutral", 2: "positive", 3: "conflict"}
text_model.config.label2id = {"negative": 0, "neutral": 1, "positive": 2, "conflict": 3}

In [988]:
polarity2id = {"negative": 0, "neutral": 1, "positive": 2, "conflict": 3}

train_texts = []
train_labels = []
train_aspects = []

for i, row in c_df.iterrows():
    text = f"{row['Sentence']}"
    aspect = f"{row['Aspect Term']}"
    train_aspects.append(aspect)
    train_texts.append(text)
    train_labels.append(polarity2id[row["polarity"]])

In [989]:
train_texts[0], train_labels[0], train_aspects[0]

('But the staff was so horrible to us', 0, 'staff')

In [979]:
train_texts[9], train_labels[9]

('meats [SEP] Our agreed favorite is the orrechiete with sausage and chicken usually the waiters are kind enough to split the dish in half so you get to sample both meats',
 1)

In [995]:
train_encodings = text_tokenizer(
    train_aspects, train_texts, truncation=True, padding=True, max_length=128
)

In [1007]:
import torch


class AspectSentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


train_dataset = AspectSentimentDataset(train_encodings, train_labels)

In [1008]:
for param in text_model.bert.parameters():
    param.requires_grad = False

In [1009]:
for param in text_model.bert.encoder.layer[-2].parameters():
    param.requires_grad = True

for param in text_model.classifier.parameters():
    param.requires_grad = True

In [1010]:
from transformers import DataCollatorWithPadding
from torch.optim import AdamW

data_collator = DataCollatorWithPadding(tokenizer=text_tokenizer)


optimizer = AdamW(
    [
        {"params": text_model.bert.encoder.layer[-1].parameters(), "lr": 1e-5},
        {"params": text_model.classifier.parameters(), "lr": 2e-5},
    ]
)

In [1016]:
text_clf_training_args = TrainingArguments(
    output_dir="./aspect_text-clg-checkpoints",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    report_to="none",
    logging_steps=500,
)

text_clf_trainer = Trainer(
    model=text_model,
    args=text_clf_training_args,
    train_dataset=train_dataset,
    tokenizer=text_tokenizer,
    data_collator=data_collator,
)

text_clf_trainer.train(resume_from_checkpoint=True)

C:\Users\pritc\AppData\Local\Temp\ipykernel_27456\2006930607.py:11: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  text_clf_trainer = Trainer(
d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
3500,0.860400
4000,0.840600
4500,0.835000
5000,0.823000


d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
d:\AI\tf-usage\lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=5260, training_loss=0.33474830946541556, metrics={'train_runtime': 1642.4778, 'train_samples_per_second': 51.197, 'train_steps_per_second': 3.202, 'total_flos': 209624186368080.0, 'train_loss': 0.33474830946541556, 'epoch': 10.0})

In [44]:
text_model.config.id2label

{0: 'negative', 1: 'neutral', 2: 'positive', 3: 'conflict'}

In [43]:
text = """I absolutely love this amazing product!”"""
asp = "meats"
inputs = text_tokenizer(
    asp, text, padding=True, truncation=True, max_length=128, return_tensors="pt"
)
probs = text_model(**inputs).logits[0]
import torch

probs = torch.softmax(probs, dim=0)
probs

tensor([0.0908, 0.0607, 0.7915, 0.0569], grad_fn=<SoftmaxBackward0>)

In [48]:
LABELS = ["Negative", "Neutral", "Positive", "Conflict"]

predictions = []
for label, prob in zip(LABELS, probs.cpu().tolist()):
    predictions.append({"label": label, "score": round(prob, 4)})

In [50]:
predictions.sort(key=lambda x: x["score"], reverse=True)

In [15]:
pip = pipeline(
    "text-classification",
    model="./Multilang-sentiment-analysis/aspect-sentimentV2-clf",
    tokenizer="./Multilang-sentiment-analysis/aspect-sentimentV2-clf",
)
pip(
    """Our agreed favorite is the orrechiete with sausage and chicken usually the waiters are kind enough to split the dish in half so you get to sample both meats"""
)

Device set to use cpu


[{'label': 'positive', 'score': 0.6320258378982544}]

In [1023]:
# text_clf_trainer.save_model(
#     r"D:\sentiment-analysis\Multilang-sentiment-analysis\aspect-sentimentV2-clf"
# )
# text_tokenizer.save_pretrained(
#     r"D:\sentiment-analysis\Multilang-sentiment-analysis\aspect-sentimentV2-clf"
# )

In [ ]:
aspect_extractor = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple",
)

aspect_sentiment = pipeline(
    "text-classification", model=text_model, tokenizer=text_tokenizer
)

Device set to use cpu


In [1087]:
def absa_pipeline(sentence):
    results = []

    # Step 1: Extract aspects
    extracted = aspect_extractor(sentence)

    # Keep only ASP entities
    aspects = [item["word"] for item in extracted if item["entity_group"] == "ASP"]

    # Remove duplicates
    aspects = list(set(aspects))

    # Step 2: Predict sentiment for each aspect
    for aspect in aspects:
        sentiment_result = aspect_sentiment(inputs=aspect, text_pair=sentence)[0]
        results.append(
            {
                "aspect": aspect,
                "sentiment": sentiment_result["label"],
                "confidence": round(sentiment_result["score"], 8),
            }
        )
    return results

In [1115]:
output = absa_pipeline("battery and screen is super")
output

[{'aspect': 'battery', 'sentiment': 'positive', 'confidence': 0.60070419},
 {'aspect': 'screen', 'sentiment': 'positive', 'confidence': 0.55145508}]

In [ ]:

from huggingface_hub import login, upload_folder

login()

upload_folder(
    folder_path="./Multilang-sentiment-analysis/aspect-token-clf/",
    repo_id="qasxasxa/aspect-extrector-Bert",
    repo_type="model",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/qasxasxa/aspect-extrector-Bert/commit/014b7af2078026621abd019b6b2d7c996484cd26', commit_message='Upload folder using huggingface_hub', commit_description='', oid='014b7af2078026621abd019b6b2d7c996484cd26', pr_url=None, repo_url=RepoUrl('https://huggingface.co/qasxasxa/aspect-extrector-Bert', endpoint='https://huggingface.co', repo_type='model', repo_id='qasxasxa/aspect-extrector-Bert'), pr_revision=None, pr_num=None)